# Lab 5.1: The Agentic Loop Lifecycle

**What you'll build:** A real agentic loop that uses `stop_reason` to drive control flow -- and learn why iteration caps and text parsing are anti-patterns.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- watch an iteration-capped loop fail on a multi-step task | 5 min |
| 2 | The Right Way -- build a stop_reason-driven agentic loop | 5 min |
| 3 | Your Turn -- extend the agent with a new tool and verify loop behavior | 10 min |
| 4 | Stress Test -- verify the agent handles edge cases correctly | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You are building an agent that can perform multi-step calculations using tools. The agent has access to a calculator tool and a memory tool (to store intermediate results).

The task: "Calculate the total cost of 3 items at $12.50 each, add 8% tax, then apply a $5 discount."

This requires multiple tool calls:
1. Multiply 3 * 12.50 = 37.50
2. Calculate tax: 37.50 * 1.08 = 40.50
3. Apply discount: 40.50 - 5.00 = 35.50

Watch what happens when the loop terminates too early.

In [ ]:
# Define simple tools for the agent
TOOLS = [
    {
        "name": "calculator",
        "description": "Performs a mathematical calculation. Pass a mathematical expression as a string.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A mathematical expression to evaluate, e.g. '3 * 12.50'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "store_result",
        "description": "Stores a named result for later reference.",
        "input_schema": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": "Name for the stored value"
                },
                "value": {
                    "type": "number",
                    "description": "The numeric value to store"
                }
            },
            "required": ["name", "value"]
        }
    }
]

# Tool execution functions
stored_values = {}

def execute_tool(name, tool_input):
    """Execute a tool and return the result."""
    if name == "calculator":
        try:
            # Safe eval for simple math expressions
            result = eval(tool_input["expression"], {"__builtins__": {}}, {})
            return json.dumps({"result": result})
        except Exception as e:
            return json.dumps({"error": str(e)})
    elif name == "store_result":
        stored_values[tool_input["name"]] = tool_input["value"]
        return json.dumps({"stored": tool_input["name"], "value": tool_input["value"]})
    return json.dumps({"error": f"Unknown tool: {name}"})

print("Tools defined: calculator, store_result")
print(f"Test: calculator('3 * 12.50') = {execute_tool('calculator', {'expression': '3 * 12.50'})}")

---

## Phase 1: The Wrong Approach

The most common mistake: using a fixed iteration cap as the primary stopping mechanism. Let's see what happens with a cap of 2 iterations on a task that needs 3+ tool calls.

In [ ]:
# ANTI-PATTERN: Fixed iteration cap as primary stopping mechanism
def run_agent_capped(user_message, max_iterations=2):
    """Agent with an iteration cap -- the WRONG way."""
    messages = [{"role": "user", "content": user_message}]
    
    print(f"Running agent with max_iterations={max_iterations}...\n")
    
    for i in range(max_iterations):
        print(f"--- Iteration {i+1}/{max_iterations} ---")
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=TOOLS,
            messages=messages
        )
        
        print(f"  stop_reason: {response.stop_reason}")
        
        # Process response content
        assistant_content = response.content
        messages.append({"role": "assistant", "content": assistant_content})
        
        # Check for tool calls
        tool_results = []
        for block in assistant_content:
            if block.type == "text":
                print(f"  Text: {block.text[:100]}..." if len(block.text) > 100 else f"  Text: {block.text}")
            elif block.type == "tool_use":
                print(f"  Tool call: {block.name}({json.dumps(block.input)})")
                result = execute_tool(block.name, block.input)
                print(f"  Tool result: {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })
        
        if tool_results:
            messages.append({"role": "user", "content": tool_results})
    
    print(f"\n=== LOOP ENDED (hit iteration cap of {max_iterations}) ===")
    print(f"stop_reason on last response: {response.stop_reason}")
    if response.stop_reason == "tool_use":
        print("WARNING: Agent wanted to continue but was cut off!")
    
    # Extract final text
    final_text = ""
    for block in response.content:
        if hasattr(block, "text"):
            final_text += block.text
    return final_text


task = "Calculate the total cost: 3 items at $12.50 each, add 8% tax, then subtract a $5 discount. Show each step."
result = run_agent_capped(task, max_iterations=2)

print(f"\n=== AGENT OUTPUT ===")
print(result if result else "(No final text -- agent was cut off mid-task)")

In [ ]:
# Score the capped agent
expected_answer = 35.50
got_answer = any(str(expected_answer) in str(result) for result in [result])

print("=== CAPPED AGENT SCORECARD ===")
print(f"  Expected final answer: ${expected_answer}")
print(f"  Got correct answer: {got_answer}")
if not got_answer:
    print("  FAILED -- The iteration cap cut off the agent before it could complete all steps.")
    print("  The agent needed 3+ tool calls but only got 2 iterations.")
    print("  This is why iteration caps should be safety nets, not primary termination.")

### Why did that fail?

- **The iteration cap of 2 cut off a 3-step task.** The model needed at least 3 tool calls but was stopped after 2.
- **The last `stop_reason` was `"tool_use"` -- the model wanted to continue.** We ignored its signal.
- **No amount of cap tuning fixes this.** You cannot predict how many steps an arbitrary task will need. Setting the cap to 100 wastes resources on simple tasks and still fails on complex ones.
- **The fundamental problem:** using iteration count as the primary control signal instead of the model's own `stop_reason`.

---

## Phase 2: The Right Approach

The correct agentic loop is `stop_reason`-driven. The model tells you when it is done via `stop_reason: "end_turn"`. Your loop continues as long as the model needs tools.

In [ ]:
def run_agent_proper(user_message, safety_cap=25):
    """
    Proper agentic loop -- stop_reason driven.
    
    The model decides when to stop via stop_reason == "end_turn".
    The safety_cap is a SAFETY NET only, not the primary mechanism.
    """
    messages = [{"role": "user", "content": user_message}]
    iteration = 0
    
    print("Running stop_reason-driven agent...\n")
    
    while True:
        iteration += 1
        
        # Safety net -- not the primary mechanism
        if iteration > safety_cap:
            print(f"\nWARNING: Hit safety cap of {safety_cap} iterations. Breaking.")
            break
        
        print(f"--- Iteration {iteration} ---")
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=TOOLS,
            messages=messages
        )
        
        print(f"  stop_reason: {response.stop_reason}")
        
        # KEY: Check stop_reason FIRST
        if response.stop_reason == "end_turn":
            # Model is done -- extract final response
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text += block.text
            print(f"\n=== AGENT FINISHED (stop_reason=end_turn, {iteration} iterations) ===")
            return final_text
        
        if response.stop_reason == "tool_use":
            # Model wants to use tools -- process them and continue
            assistant_content = response.content
            messages.append({"role": "assistant", "content": assistant_content})
            
            tool_results = []
            for block in assistant_content:
                if block.type == "text":
                    print(f"  Text: {block.text[:100]}")
                elif block.type == "tool_use":
                    print(f"  Tool call: {block.name}({json.dumps(block.input)})")
                    result = execute_tool(block.name, block.input)
                    print(f"  Tool result: {result}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            
            messages.append({"role": "user", "content": tool_results})
            continue  # Loop back -- model decides next action
    
    return "(Agent hit safety cap)"


# Run the same task with the proper agent
result = run_agent_proper(task)

print(f"\n=== AGENT OUTPUT ===")
print(result)

In [ ]:
# Compare results
print("=" * 55)
print("COMPARISON: CAPPED vs PROPER")
print("=" * 55)
print(f"{'Metric':<30} {'Capped':>10} {'Proper':>10}")
print(f"{'-'*30} {'-'*10} {'-'*10}")
print(f"{'Termination signal':<30} {'iter cap':>10} {'stop_reason':>10}")
print(f"{'Completed task':<30} {'No':>10} {'Yes':>10}")
print(f"{'Model-driven':<30} {'No':>10} {'Yes':>10}")
print()
print("The proper agent let the model decide when to stop.")
print("It used exactly as many iterations as needed -- no more, no less.")

---

## Phase 3: Your Turn

Extend the agent with a new tool and verify the agentic loop handles it correctly.

**Your task:** Add a `lookup_price` tool that retrieves product prices from a catalog. Then ask the agent to look up prices for multiple items, calculate the total, and store it.

This tests that your loop handles an unpredictable number of tool calls.

In [ ]:
# Product catalog for the lookup tool
CATALOG = {
    "widget": 12.50,
    "gadget": 24.99,
    "doohickey": 7.75,
    "thingamajig": 31.00,
    "whatchamacallit": 15.25,
}

# =============================================================
# TODO: Add the lookup_price tool to the TOOLS list and
# add its execution logic to execute_tool.
# =============================================================
#
# The tool should:
# - Accept a "product_name" string parameter
# - Return the price from CATALOG, or an error if not found
#
# Then run the agent with this task:
# "Look up prices for widget, gadget, and doohickey. Calculate
#  the total cost of 2 widgets, 1 gadget, and 3 doohickeys.
#  Add 8% tax. Store the final total."

EXTENDED_TOOLS = TOOLS + [
    # TODO: Add your lookup_price tool definition here
    # {
    #     "name": "lookup_price",
    #     "description": "...",
    #     "input_schema": { ... }
    # }
]

def execute_tool_extended(name, tool_input):
    """Extended tool executor with lookup_price."""
    if name == "lookup_price":
        # TODO: Implement lookup logic
        # product = tool_input["product_name"].lower()
        # if product in CATALOG:
        #     return json.dumps({"product": product, "price": CATALOG[product]})
        # return json.dumps({"error": f"Product '{product}' not found in catalog"})
        pass
    return execute_tool(name, tool_input)

print("Catalog loaded:", CATALOG)
print("\nTODO: Add the lookup_price tool and run the agent.")

In [ ]:
# =============================================================
# Checker: validates your extended agent
# =============================================================

# Expected: 2*12.50 + 1*24.99 + 3*7.75 = 25.00 + 24.99 + 23.25 = 73.24
# With 8% tax: 73.24 * 1.08 = 79.0992 ≈ 79.10
expected_total = round(73.24 * 1.08, 2)

print(f"Expected total (with tax): ${expected_total}")
print(f"Stored values: {stored_values}")

if stored_values:
    stored_total = list(stored_values.values())[-1]  # Last stored value
    if abs(stored_total - expected_total) < 0.05:
        print(f"\nPASSED -- Agent correctly computed and stored ${stored_total:.2f}")
    else:
        print(f"\nFAILED -- Expected ~${expected_total}, got ${stored_total:.2f}")
else:
    print("\nFAILED -- No values stored. Complete the TODO and run the agent.")

---

## Phase 4: Stress Test

Let's verify the agent handles edge cases correctly with the proper agentic loop.

In [ ]:
# Test 1: Simple task (should complete in 1-2 iterations)
print("=== Test 1: Simple task ===")
simple_result = run_agent_proper("What is 42 * 17? Use the calculator.")
print(f"Result: {simple_result}")
print()

# Test 2: Task with no tool use needed (should complete in 1 iteration)
print("=== Test 2: No tools needed ===")
no_tool_result = run_agent_proper("What is the capital of France? (Do not use any tools.)")
print(f"Result: {no_tool_result}")
print()

print("=== STRESS TEST RESULTS ===")
print("The proper agentic loop adapts to each task:")
print("  - Multi-step tasks: runs as many iterations as needed")
print("  - Simple tasks: completes quickly")
print("  - No-tool tasks: exits immediately on end_turn")

### Key Takeaways

1. **The agentic loop is `stop_reason`-driven.** `while True` + check `stop_reason` is the correct pattern. The model signals completion, not your code.
2. **Iteration caps are safety nets, not control flow.** Set a high cap (25+) as a safety net, but never as the primary termination mechanism.
3. **Never parse text for termination.** The model can emit text AND tool_use in the same response. Only `stop_reason` is reliable.
4. **The agent adapts to task complexity.** Simple tasks use few iterations, complex tasks use many -- the loop handles both correctly.